In [ ]:
"""
FILE 5: DEQUE, HEAPQ, BISECT, SORTED CONTAINERS
===================================================
Yeh sab special data structures hain jo BFS, priority queue, aur
binary-search-on-sorted-data jaise problems ke liye zaroori hain.
"""

from collections import deque
from heapq import heappush, heappop, heapify, nlargest, nsmallest, heappushpop, heapreplace
from bisect import bisect_left, bisect_right, insort


# ===============================================================
# 1) DEQUE - Double Ended Queue (dono taraf se O(1) add/remove)
# ===============================================================
dq = deque()
dq.append(1)          # right mein add        -> O(1)
dq.pop()                # right se remove       -> O(1)
dq.appendleft(1)         # LEFT mein add          -> O(1)  <- yeh deque ki khaasiyat hai
dq.popleft()               # LEFT se remove          -> O(1)
peek_left = dq[0] if dq else None    # left dekhna (peek)
peek_right = dq[-1] if dq else None   # right dekhna (peek)

# NORMAL list ka lst.pop(0) O(n) hota hai (SLOW), isliye BFS mein deque use karo!

# -------------------------------------------------------------
# BFS TEMPLATE - grid pe (yeh yaad rakho, baar baar aata hai)
# -------------------------------------------------------------
def bfs_grid(start_row, start_col, rows, cols, directions):
    queue = deque([(start_row, start_col)])
    visited = set()
    visited.add((start_row, start_col))

    while queue:
        row, col = queue.popleft()   # O(1) -> FIFO order mein nikalo
        # yahan current cell process karo
        for dr, dc in directions:
            nr, nc = row + dr, col + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc))   # O(1) -> peeche jodo

# LEVEL-ORDER BFS (jab tumhe pata hona chahiye ki kaunsa node kis "level" pe hai):
def bfs_level_order(start, graph):
    visited = {start}
    queue = deque([start])
    level = 0
    while queue:
        size = len(queue)   # PEHLE current level ka size le lo
        for _ in range(size):
            node = queue.popleft()
            # process node (jo isi level pe hai)
            for neighbor in graph[node]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        level += 1   # ab agla level shuru hoga


# -------------------------------------------------------------
# MONOTONIC DEQUE - "sliding window maximum/minimum" wale problems ke liye
# -------------------------------------------------------------
def maxSlidingWindow(nums, k):
    dq2 = deque()   # yahan index store honge, values DECREASING order mein rahengi
    result = []

    for i in range(len(nums)):
        # window ke bahar wale index hatao
        while dq2 and dq2[0] < i - k + 1:
            dq2.popleft()

        # chhote elements ko hata do - wo kabhi max nahi ban sakte
        while dq2 and nums[dq2[-1]] < nums[i]:
            dq2.pop()

        dq2.append(i)

        if i >= k - 1:
            result.append(nums[dq2[0]])   # front hamesha max hoga

    return result

# Bounded deque (fixed maxsize) - full hote hi dusri taraf se khud remove ho jaata hai:
window = deque(maxlen=3)
window.append(1)   # deque([1])
window.append(2)    # deque([1, 2])
window.append(3)     # deque([1, 2, 3])
window.append(4)      # deque([2, 3, 4]) -> 1 khud-ba-khud nikal gaya


# ===============================================================
# 2) HEAPQ - Python ka MIN-HEAP (sabse chhota element hamesha top pe)
# ===============================================================
heap = []
heappush(heap, 5)     # O(log n) insert
heappush(heap, 2)
heappush(heap, 8)
smallest = heappop(heap)   # O(log n) -> smallest nikalke return karta hai
peek = heap[0] if heap else None   # O(1) -> bina nikale dekhna

nums_h = [5, 2, 8, 1, 9]
heapify(nums_h)   # O(n) -> existing list ko heap mein convert karna
                    # (n baar heappush karne se yeh FAST hai)

# -------------------------------------------------------------
# MAX-HEAP TRICK - Python sirf min-heap deta hai, max-heap ke liye NEGATE karo
# -------------------------------------------------------------
max_heap = []
heappush(max_heap, -5)   # value ko NEGATIVE karke daalo
heappush(max_heap, -2)
heappush(max_heap, -8)
largest = -heappop(max_heap)   # nikalte waqt WAPAS negate karo
# Kaam kaise karta hai: agar a < b hai, to -a > -b hoga, isliye order ulat jaata hai.

# -------------------------------------------------------------
# TUPLES HEAP MEIN - multi-key sort ke liye
# -------------------------------------------------------------
# Tuple ka PEHLA element primary sort key banta hai:
pq = []
heappush(pq, (0, "start_node"))       # Dijkstra: (distance, node)
dist, node = heappop(pq)               # sabse kam distance wala pehle aayega

pq2 = []
heappush(pq2, (5, -100, "task1"))      # (deadline, -profit, task_id) -> ties profit se break

# SAWDHAAN: agar distance same hai aur dusra element compare NAHI ho sakta
# (jaise custom object bina __lt__ ke), to Python TypeError dega.
# Isliye HAMESHA ek tie-breaker (counter) daalo beech mein:
counter = 0
heap3 = []
heappush(heap3, (5, counter, "some_object"))   # counter ties ko safely break karega
counter += 1


# -------------------------------------------------------------
# HELPFUL HEAP FUNCTIONS
# -------------------------------------------------------------
nums_list = [9, 3, 7, 1, 5]
print(nlargest(3, nums_list))    # top-3 bade elements - sort kiye bina O(n log k)
print(nsmallest(3, nums_list))    # top-3 chhote elements

result_push_pop = heappushpop(heap, 4)   # push karo phir pop karo (ek call mein, efficient)
result_replace = heapreplace(heap, 4)      # pop karo phir push karo

# LIMITATION: heapq mein beech ka koi specific element seedha remove nahi kar sakte
# (O(n) scan lagega). Agar baar baar element invalidate/remove karna ho, to:
#   - "lazy deletion" trick use karo (mark as deleted, jab top pe aaye skip karo)
#   - ya phir sortedcontainers ki SortedList use karo (O(log n) removal deti hai)


# ===============================================================
# 3) BISECT - sorted list mein binary search
# ===============================================================
sorted_list = [1, 3, 5, 7, 9]

print(bisect_left(sorted_list, 5))    # 2 -> 5 ki LEFTMOST insert position
print(bisect_right(sorted_list, 5))    # 3 -> 5 ki RIGHTMOST insert position
print(bisect_left(sorted_list, 4))      # 2 -> 4 kahan insert hoga
print(bisect_left(sorted_list, 0))       # 0 -> shuru mein
print(bisect_left(sorted_list, 10))       # 5 -> end mein

# DIFFERENCE samjho:
#   bisect_left  -> "pehla element jo target se BADA-YA-BARABAR hai" uska index
#   bisect_right -> "pehla element jo target se BADA hai (strictly)" uska index

# FLOOR (sabse bada element jo target se chhota ya barabar hai):
target = 6
i = bisect_right(sorted_list, target) - 1
floor_val = sorted_list[i] if i >= 0 else None

# CEILING (sabse chhota element jo target se bada ya barabar hai):
i = bisect_left(sorted_list, target)
ceiling_val = sorted_list[i] if i < len(sorted_list) else None

# Range [lo, hi] mein kitne elements hain:
lo, hi = 2, 8
count_in_range = bisect_right(sorted_list, hi) - bisect_left(sorted_list, lo)

# Sorted order maintain karte hue insert karna:
insort(sorted_list, 4)   # list ban jaayegi: [1, 3, 4, 5, 7, 9]
# NOTE: insort ka search part O(log n) hai, par insertion (shifting) O(n) hai.
# Agar bahut baar insert karna ho large list mein, to SortedList use karo.


# ===============================================================
# 4) SORTEDCONTAINERS - SortedList, SortedDict, SortedSet (O(log n) operations)
# ===============================================================
# Yeh Python ki standard library mein NAHI hai, par LeetCode aur bahut se
# platforms pe available hai. Balanced BST jaisa kaam deti hai.

# from sortedcontainers import SortedList, SortedDict, SortedSet

# sl = SortedList([5, 1, 3, 1])   # SortedList([1, 1, 3, 5])
# sl.add(2)                         # O(log n) insert
# sl.remove(3)                       # O(log n) remove (missing ho to ValueError)
# sl.discard(99)                       # O(log n) remove (missing ho to error nahi)
# sl[0]                                  # smallest element
# sl[-1]                                   # largest element
# sl.bisect_left(3)                          # index dhundo
# sl.irange(2, 5)                              # [2, 5] range ke saare elements

# NEIGHBOR QUERIES (yeh formulas yaad rakh lo, interviews mein kaam aate hain):
#   Floor (sabse bada <= x)        -> sl[sl.bisect_right(x) - 1]   (pehle index>=0 check karo)
#   Ceiling (sabse chhota >= x)     -> sl[sl.bisect_left(x)]        (pehle index<len(sl) check karo)
#   Strictly lower (sabse bada < x)  -> sl[sl.bisect_left(x) - 1]
#   Strictly higher (sabse chhota>x)  -> sl[sl.bisect_right(x)]